### Silver - Canais

Problemas identificados:
- `CH05` duplicado, com os valores "E-commerce" e "ecommerce" — mesmo
  canal, grafias distintas
- `CH06` sem nome cadastrado (a planilha de origem já traz a observação
  "nome ausente")
- `ch07` com identificador em caixa baixa, mesmo padrão observado em
  `vendedores.csv`
- campo `ativo` com valores `sim`/`Sim`/`SIM`/`nao`

A resolução do conflito em `CH05` não pode ser feita por ordenação
lexicográfica nem por `initcap()`: ordenar por nome decrescente assumindo
"E-commerce" > "ecommerce" é incorreto, pois maiúsculas precedem
minúsculas na ordenação ASCII; e `initcap()` não resolve a divergência
porque a diferença entre os dois valores está na presença do hífen, não
na capitalização. O valor de referência para `CH05` é definido
explicitamente após a deduplicação.

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_canais_bronze = spark.table(f"{schema_name}.bronze_canais")
display(df_canais_bronze)

In [ ]:
df_canais_step1 = (
    df_canais_bronze
    .withColumn("id_canal", F.upper(F.trim(F.col("id_canal"))))
    .withColumn("tipo_canal", F.initcap(F.trim(F.col("tipo_canal"))))
    .withColumn(
        "ativo_bool",
        F.when(F.lower(F.trim(F.col("ativo"))) == "sim", True)
         .when(F.lower(F.trim(F.col("ativo"))) == "nao", False)
         .otherwise(None)
    )
)

In [ ]:
w_dedup = Window.partitionBy("id_canal").orderBy(F.monotonically_increasing_id())

df_canais_dedup = (
    df_canais_step1
    .withColumn("rk", F.row_number().over(w_dedup))
    .filter(F.col("rk") == 1)
    .drop("rk")
    .withColumn(
        "nome_canal",
        F.when(F.col("id_canal") == "CH05", F.lit("E-commerce"))
         .otherwise(F.initcap(F.trim(F.col("nome_canal"))))
    )
)

O canal `CH06` não possui nome cadastrado. O registro é mantido — a
exclusão removeria receita associada a pedidos vinculados a esse canal —
e recebe um nome genérico, sinalizado pela coluna `nome_canal_ausente`.

In [ ]:
df_canais_silver = (
    df_canais_dedup
    .withColumn("nome_canal_ausente", F.col("nome_canal").isNull())
    .withColumn(
        "nome_canal",
        F.when(F.col("nome_canal").isNull(), F.concat(F.lit("Canal não identificado ("), F.col("id_canal"), F.lit(")")))
         .otherwise(F.col("nome_canal"))
    )
    .select(
        F.col("id_canal").alias("canal_id"),
        "nome_canal",
        "tipo_canal",
        "ativo_bool",
        "nome_canal_ausente",
        F.col("observacao").alias("observacao_origem"),
    )
)

(
    df_canais_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.silver_canais")
)

display(df_canais_silver)